#Step 1: Import data
BUS 3392A Group Project: Steam Games Data
Team: Josh Baron and Juan Haro

In [2]:
!pip install kagglehub

In [3]:
import kagglehub

100%|██████████| 204M/204M [00:01<00:00, 162MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/hubertsidorowicz/steam-games-dataset-daily-updates/versions/115


## Step 2: Load only the columns we need
The full file has more than 40 columns. We load only the 12 the assignment requires so the table is smaller, loads faster, and is easier to read. `app_id` is the unique ID for each game, so we use it whenever we count games.

In [9]:
import pandas as pd

columns = ["app_id", "name", "release_date", "price", "price_status",
           "estimated_owners", "genres", "categories",
           "positive", "negative", "recommendations", "average_playtime_forever"]

gamesData_location = kagglehub.dataset_download("hubertsidorowicz/steam-games-dataset-daily-updates/versions/114")
gamesData = pd.read_csv(gamesData_location + "/steam_games.csv", usecols=columns, low_memory=False)

100%|██████████| 204M/204M [00:01<00:00, 175MB/s]

Extracting files...


## Step 3: Check data quality
Before cleaning, we checked the size of the data, blanks, and duplicates. Version 114 has
150,643 games. What we found:
- `name`: 16 blanks
- `release_date`: 82 blanks
- `price`: 1,425 blanks (games that are no longer for sale)
- `recommendations`: 52,959 blanks
- `genres` / `categories`: 8,434 and 8,962 games have an empty list, stored as the text `"[]"`
- No duplicate `app_id` values, so each row is one unique game

Other issues to fix in later steps: `release_date` is stored as text, and
`estimated_owners` comes in two different formats.

In [10]:
gamesData.head()

,app_id,name,release_date,price,price_status,estimated_owners,genres,categories,positive,negative,recommendations,average_playtime_forever
0,1265390,EllrLand,2020-04-28,0.0,free,0 - 20000,"[""Adventure"", ""Strategy""]","[""Single-player"", ""Family Sharing""]",38,28,0.0,0
1,1266480,MiNNaDe Counter Fight,2020-05-01,0.0,free,0 - 20000,"[""Action"", ""Casual"", ""Indie"", ""Early Access""]","[""Single-player"", ""Multi-player"", ""Co-op"", ""Sh...",12,6,0.0,0
2,1289500,BattleParty,2020-04-27,0.0,free,0 - 20000,"[""Action"", ""Adventure"", ""Free To Play"", ""Indie...","[""Single-player"", ""Multi-player"", ""Co-op"", ""LA...",17,9,0.0,0
3,1318140,BookConverter,2020-05-15,0.0,free,0 - 0,[],"[""Family Sharing""]",0,0,0.0,0
4,1342260,SAMURAI SHODOWN,2021-06-14,0.0,free,0 - 0,"[""Action""]","[""Single-player"", ""Multi-player"", ""PvP"", ""Onli...",0,0,1587.0,0


In [11]:
gamesData.shape

(150643, 12)

In [12]:
gamesData.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150643 entries, 0 to 150642
Data columns (total 12 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   app_id                    150643 non-null  int64  
 1   name                      150627 non-null  object 
 2   release_date              150561 non-null  object 
 3   price                     149218 non-null  float64
 4   price_status              150643 non-null  object 
 5   estimated_owners          150643 non-null  object 
 6   genres                    150643 non-null  object 
 7   categories                150643 non-null  object 
 8   positive                  150643 non-null  int64  
 9   negative                  150643 non-null  int64  
 10  recommendations           97684 non-null   float64
 11  average_playtime_forever  150643 non-null  int64  
dtypes: float64(2), int64(4), object(6)
memory usage: 13.8+ MB


In [13]:
gamesData["app_id"].duplicated().sum()

np.int64(0)

In [14]:
print((gamesData["genres"] == "[]").sum())
print((gamesData["categories"] == "[]").sum())

8434
8962


In [15]:
#Part 2: Missing Data
gamesData.isna().sum()

,0
app_id,0
name,16
release_date,82
price,1425
price_status,0
estimated_owners,0
genres,0
categories,0
positive,0
negative,0


In [16]:
gamesData["app_id"].duplicated().sum()

np.int64(0)

In [17]:
print((gamesData["genres"] == "[]").sum())
print((gamesData["categories"] == "[]").sum())

8434
8962


#Remove games with no name or release date
We removed games missing a `name` or `release_date`. Without a name we can't identify the game, and without a date we can't use it to compare older and newer games (Q3). These are fewer than 100 games, so removing them barely changes the dataset.

We kept games missing `price`, `recommendations`, or genres. Dropping them here would throw away tens of thousands of games that are still useful for other questions.

In [18]:
print("Games before:", len(gamesData))

Games before: 150643


In [19]:
gamesData = gamesData.dropna(subset=["name", "release_date"])

In [20]:
print("Games after:", len(gamesData))

Games after: 150559


In [21]:
gamesData.isna().sum()

,0
app_id,0
name,0
release_date,0
price,1374
price_status,0
estimated_owners,0
genres,0
categories,0
positive,0
negative,0


#Convert release dates to real dates
`release_date` was stored as text (for example `"2020-01-21"`), so Python can't sort it by time or pull out the year. `pd.to_datetime` turns it into real dates.

In [22]:
gamesData["release_date"] = pd.to_datetime(gamesData["release_date"])

In [23]:
print(gamesData["release_date"].dtype)
print(gamesData["release_date"].isna().sum())
print(gamesData["release_date"].min(), gamesData["release_date"].max())

datetime64[ns]
0
1997-06-30 00:00:00 2026-11-21 00:00:00


## Step 6: Add release year and month
For Q3 (older vs. newer games), we need to group games by when they came out.
We created two new columns, `release_year` and `release_month`, from the converted dates.

In [24]:
gamesData["release_year"] = gamesData["release_date"].dt.year
gamesData["release_month"] = gamesData["release_date"].dt.month
gamesData[["release_date", "release_year", "release_month"]].head()

,release_date,release_year,release_month
0,2020-04-28,2020,4
1,2020-05-01,2020,5
2,2020-04-27,2020,4
3,2020-05-15,2020,5
4,2021-06-14,2021,6


Step 7:
Estimated owners is written in two different formats.
Changing it to one standard format.

In [25]:
gamesData["estimated_owners"] = gamesData["estimated_owners"].str.replace(",", "")
gamesData["estimated_owners"] = gamesData["estimated_owners"].str.replace(" .. ", " - ")
gamesData["estimated_owners"].value_counts()

,count
estimated_owners,
0 - 20000,103846
0 - 0,21084
20000 - 50000,11415
50000 - 100000,5371
100000 - 200000,3469
200000 - 500000,2861
500000 - 1000000,1161
1000000 - 2000000,727
2000000 - 5000000,404


Step 8: Convert estimated owners to a singular number using a midpoint of the range. We created a new column called 'owners_midpoint' to reflect this.

In [26]:
owners_numbers = []

for value in gamesData["estimated_owners"]:
    if pd.isna(value):
        owners_numbers.append(None)
    elif value == "0 - 0":
        owners_numbers.append(None)
    else:
        low, high = value.split(" - ")
        owners_numbers.append((int(low) + int(high)) / 2)

gamesData["owners_midpoint"] = owners_numbers
gamesData[["estimated_owners", "owners_midpoint"]].head()

,estimated_owners,owners_midpoint
0,0 - 20000,10000.0
1,0 - 20000,10000.0
2,0 - 20000,10000.0
3,0 - 0,NaN
4,0 - 0,NaN


In [28]:
gamesData.to_csv("steam_clean.csv", index=False)